In [ ]:
import torch
from torch.utils.data import Dataset, random_split 

from collections import Counter
import pandas as pd
import numpy as np
import random
import os
import json
from sklearn.metrics import f1_score
import optuna
import shutil
from datetime import datetime
import transformers
import wandb
os.environ["WANDB_WATCH"]  = "all"
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.cm as cm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import sys
sys.path.append('../')
from src.database import *
from src.probing import *


save_path = "../data/"
    
for directory in [save_path + "probing_results", save_path + "checkpoints", save_path + "sampling"]:
    if not os.path.exists(directory):
        os.makedirs(directory)
    
    
def save_gpu_training_time(training_start, file):
    
    with open(file, "a") as f:
        
        f.write(str((datetime.now() - training_start).total_seconds()))
        f.write("\n")

## Getting the data

In [ ]:

datasets = ["semcor&omsti_noun-synsets-strat", "semcor&omsti_person.n.01", "cctweets-random", "cctweets-activist"]
models = ["bert-base-uncased", "microsoft-deberta-base"] 
layers = [6, 12]
output = ["kmeans2", "kmeans5", "pc1", "pc2"]


files = []

for file in os.listdir(save_path + "clustering")  + os.listdir(save_path + "principal_components"):
    
    if ".npy" in file:
    
        dataset_name, model_name, layer, filter_name, target_info = file.split("_")
        
        if filter_name != "None":
            
            dataset_name = "_".join([dataset_name, filter_name])
    
        if dataset_name in datasets and model_name in models and target_info.split("-")[0] in output:
            
            files.append(file)
            
# Loading available results
if not os.path.isfile(save_path + "/probing_results/probing_results.csv"):
    result_df = pd.DataFrame()
else:
    result_df = pd.read_csv(save_path + "/probing_results/probing_results.csv", index_col = 0)

result_df.head()

In [ ]:
# database test

file = files[0]
file_path = save_path + "clustering/" + file
output_type = "cluster_prediction"

print(file)
      
dataset_name, model_name, layer, filter_name, target_info = file.split("_")

df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)

j = np.random.choice(range(0,len(df)), 1).item()


for input_type in["bow_simple", "syntax", "position"]:

    db = build_db(file_path, input_type, output_type, df, sample_size = None)

    print("-"*50)
    print(input_type)
    print(db[j])
    print(db.target_dim)
    print(db.lol_tokens[j])
    print(db.tensor_to_lol_tokens(torch.nonzero(db[j]["input"]))) if "bow" in input_type else print(db.tensor_to_lol_tokens(db[j]["input"][torch.nonzero(db[j]["input"])]))
    print(df.loc[j, "text_masked"])

## Functions

In [ ]:
# model init function arguments could not be overwritten, thus variables need to be global
def model_init():
    
    n_tokens = db.n_tokens
    n_hidden = args.n_hidden
    n_hidden_layers = args.n_hidden_layers
    input_type = args.input_type
    target_dim = db.target_dim
    
    hidden_layers = nn.ModuleList()
        
    for n in range(n_hidden_layers):
        hidden_layers.append(nn.Linear(n_hidden,n_hidden))
        hidden_layers.append(nn.ReLU())

            
    if "bow" in input_type:
        
        model = nn.Sequential(
            nn.Linear(n_tokens,n_hidden),
            nn.ReLU(),
            nn.Sequential(*hidden_layers) ,
            nn.Linear(n_hidden, target_dim))
        
    elif input_type == "syntax":
        
        syntax_embedder = SyntaxEmbedding(n_tokens, db.max_len, n_hidden)

        model = nn.Sequential(
            syntax_embedder,
            nn.ReLU(),
            nn.Sequential(*hidden_layers) ,
            nn.Linear(n_hidden, target_dim)
            )
        
    elif input_type == "position":
        
        model = nn.Sequential(
            nn.Linear(db.max_len, n_hidden),
            nn.ReLU(),
            nn.Sequential(*hidden_layers) ,
            nn.Linear(n_hidden, target_dim)
            )
        
    return model

# Hyperparameter Search
### Defining the search space

In [ ]:
if not os.path.isfile(save_path + "hyperparameter_search.csv"):
    hp_search = pd.DataFrame()
else:
    hp_search = pd.read_csv(save_path + "hyperparameter_search.csv", index_col = 0)
    


# define search space
def my_hp_space(trial):
    return {
        
        # parameters to model_init
        "n_hidden": trial.suggest_int("n_hidden",128, 2048, log= True),
        
        # regular training arguments
        "max_steps": trial.suggest_int("max_steps", 5000, 50000, log=True),
        "per_device_train_batch_size": trial.suggest_int("per_device_train_batch_size", 8, 64, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 0.1, log=True)
        
    }

def my_objective(metrics):
    
    # ojective to optimize with hyperparameter search is set to Macro F1
    return metrics["eval_macro_f1"] if "eval_macro_f1" in metrics.keys() else metrics["eval_R²"]


hp_search

### Running Hyperparameter Search

In [ ]:
for input_type in ["syntax", "position", "bow_simple"]:
    
    sample_files = []

    for name in ["noun-synsets-strat", "person.n.01", "cctweets-random", "cctweets-activist"]:
    
        for o in ["kmeans", "pc"]:
        
            candidate_files = [file for file in files if name in file and o in file and not "layer0" in file]

            sample_files.extend([file for file in random.sample(candidate_files, k = 3)])

            
    for file in sample_files:
        
        print("-"*100)
        print(input_type)
        print("-"*100)
        print(file)
        
        dataset_name, model_name, layer, filter_name, target_info = file.split("_")
        df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)
        df = df.loc[df[filter_name]== True] if filter_name != "None" else df
        output_type = "cluster_prediction" if "kmeans" in file else "pc_regression" 
        file_path = save_path + "clustering/" + file if "kmeans" in file else lc_path + "principal_components/" + file
        db = build_db(file_path, input_type, output_type, df, sample_size = None)
        
        # calculate label weights for loss function
        if output_type == "cluster_prediction":
            cluster_counts = Counter(db.targets)
            class_weights = {
                "equal": [1. for cluster in range(0, db.target_dim)],
                "balanced": [max(cluster_counts.values())/ cluster_counts[cluster] if cluster_counts[cluster]>0 else 0 for cluster in range(0,db.target_dim)]
            }
        
        # split database
        test_length = round(0.15*(len(db)))
        eval_length = round(0.15*(len(db)))
        train_length = len(db)-eval_length-test_length
        
        train_dataset, eval_dataset, test_dataset = random_split(db, [train_length, eval_length, test_length], 
                                               generator=torch.Generator().manual_seed(42))

        
    
        # define training arguments
        args = MyTrainingArguments(
            save_path + "hp_search_checkpoints",
            overwrite_output_dir = True,
            evaluation_strategy = "steps",
            max_grad_norm=1,
            logging_first_step = True,
            lr_scheduler_type = "linear",
            logging_steps = 500,
            per_device_eval_batch_size = 64,
            seed = 42,
            save_steps = 10000000,
            warmup_steps = 500,
            save_total_limit = 0,
            #run_name=clustering_file.split(".")[0],  # for using wandb
            report_to=[],  # wandb or tensorboard
            input_type = input_type,
            output_type = output_type,
            target_dim = db.target_dim,
            dataloader_drop_last = False,
            class_weights=torch.tensor(class_weights["balanced"]).to(device) if output_type == "cluster_prediction" else None, # equal
            weight_decay=0.1,
            n_hidden_layers=1
        )
        
         # define trainer
        if db.output_type == "cluster_prediction": 
            trainer = MyClassificationTrainer(
                model_init=model_init,
                args=args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                compute_metrics=compute_classification_metrics
                )  
        else:
            trainer = MyRegressionTrainer(
            model_init=model_init,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            compute_metrics=compute_regression_metrics)
        
        # run search
        training_start = datetime.now()
        best_run = trainer.hyperparameter_search(direction="maximize", n_trials = 50, hp_space=my_hp_space,
                                                compute_objective= my_objective, 
                                                study_name = f'{file}_{input_type}')
        save_gpu_training_time(training_start, f"{save_path}training_time_hp_search.txt")
        
        
        hp = best_run[2]
        
        hp["dataset"]= dataset_name
        hp["model"] = model_name
        hp["layer"] = layer
        hp["filter"] = filter_name
        hp["output_type"] = target_info.split("-")[0]
        hp["input_type"] = input_type
        hp["objective_name"] = "Macro-F1" if db.output_type == "cluster_prediction" else "R²"
        hp["objective"] = best_run.objective
        hp["datetime"] = datetime.now().strftime("%d.%m.%Y %H:%M")
        
        hp_search = hp_search.append(hp, ignore_index = True)
        hp_search.to_csv(save_path + "hyperparameter_search.csv")
 

In [ ]:
# saving hyperparameter search statistics
hp_search["output_type_"] = hp_search["output_type"].str.replace("[0-9]", "")
hp_search_summary = hp_search[["dataset", "model","filter","input_type", "output_type_", "n_hidden", "max_steps", "per_device_train_batch_size", "learning_rate", "objective"]].groupby(["dataset","filter","input_type", "output_type_"]).describe()
hp_search_summary.to_csv(save_path + "hp_search_summary.csv")
hp_search_summary["max_steps"]


In [ ]:
# specifying and saving hyperparameter settings

hp_settings = pd.DataFrame(columns = ["dataset","filter", "input_type", "output_type", 
                                      "n_hidden","per_device_train_batch_size", "learning_rate", "max_steps"])
for idx in hp_search_summary.index:
    
    row = hp_search_summary.loc[idx]
    
    settings = {"dataset": idx[0],
               "filter": idx[1],
               "input_type": idx[2],
               "output_type": idx[3],
               "n_hidden": int(row["n_hidden", "max"]),
               "per_device_train_batch_size":  int(round(row["per_device_train_batch_size", "mean"])),
               "learning_rate": row["learning_rate", "mean"],
               "max_steps": int(row["max_steps", "max"] + 5000)
               
    }

    hp_settings = hp_settings.append(settings, ignore_index = True)
    
hp_settings.to_csv(save_path + "hyperparameter_settings.csv")

hp_settings.set_index(["dataset", "filter", "input_type", "output_type"], inplace = True)

# Run full cluster classifications

In [ ]:
def count_parameters(model):   
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

stats = pd.DataFrame()

In [ ]:
for file in files:
    
    print("\n")
    print(file)
    print("-"*100)
    
    dataset_name, model_name, layer, filter_name, target_info = file.split("_")
    config_name = "_".join([dataset_name, model_name, layer, filter_name, target_info.split("-")[0]])
    df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)
    df = df.loc[df[filter_name]== True] if filter_name != "None" else df
    output_type = "cluster_prediction" if "kmeans" in file else "pc_regression"
    file_path = save_path + "clustering/" + file if "kmeans" in file else lc_path + "principal_components/" + file
        
    
    for input_type in ["bow_simple", "position", "syntax"]: #, f"bow_{model_name}"
        
        
        if len (result_df) > 0 and len(result_df.loc[(result_df["Dataset"] == dataset_name) & 
                            (result_df["Model"] == model_name) &
                            (result_df["Layer"] == layer) & 
                             (result_df["Filter"] == filter_name) &
                             (result_df["Input"] == input_type) &
                             (result_df["Output"] == target_info.split("-")[0])]) != 0:
               continue
        

        print(input_type)
        print("-"*100)
        
        db = build_db(file_path, input_type, output_type, df, sample_size = None)
        
        # calculate label weights for loss function
        if output_type == "cluster_prediction":
            cluster_counts = Counter(db.targets)
            class_weights = {
                "equal": [1. for cluster in range(0, db.target_dim)],
                "balanced": [max(cluster_counts.values())/ cluster_counts[cluster] if cluster_counts[cluster]>0 else 0 for cluster in range(0,db.target_dim)]
            }
        
        # split database
        test_length = round(0.15*(len(db)))
        eval_length = round(0.15*(len(db)))
        train_length = len(db)-eval_length-test_length
        
        train_dataset, eval_dataset, test_dataset = random_split(db, [train_length, eval_length, test_length], 
                                               generator=torch.Generator().manual_seed(42))


        if not os.path.isfile(f"{save_path}sampling/{dataset_name}-{filter_name}_train-idx.npy"):
            np.save(f"{save_path}sampling/{dataset_name}-{filter_name}_train-idx.npy", train_dataset.indices)
            np.save(f"{save_path}sampling/{dataset_name}-{filter_name}_eval-idx.npy", eval_dataset.indices)
            np.save(f"{save_path}sampling/{dataset_name}-{filter_name}_test-idx.npy", test_dataset.indices)
        
        # retrieving hyperparameter settings
        idx = [dataset_name, filter_name, input_type, "kmeans" if output_type == "cluster_prediction" else "pc"]
        hp_setting = hp_settings.loc[[idx]]
            
            
        # defining training arguments
        args = MyTrainingArguments(
            save_path + "checkpoint",
            overwrite_output_dir = False,
            evaluation_strategy = "steps",
            max_grad_norm=1,
            lr_scheduler_type = "linear",
            logging_steps = 500,
            save_steps = 500,
            seed = 42,
            run_name=config_name,  # for using wandb
            report_to=["wandb"],
            input_type = input_type,
            output_type = output_type,
            target_dim = db.target_dim,
            gradient_accumulation_steps=1,
            warmup_steps=500,
            per_device_eval_batch_size=64,
            class_weights=torch.tensor(class_weights["balanced"]).to(device) if output_type == "cluster_prediction" else None, # equal
            weight_decay=0.1,
            n_hidden_layers=1,
            
            # parameters determined in hyperparameter search
            n_hidden=int(hp_setting["n_hidden"][0]),
            per_device_train_batch_size=int(hp_setting["per_device_train_batch_size"][0]),
            learning_rate=hp_setting["learning_rate"][0],
            max_steps = int(hp_setting["max_steps"][0]),
            
            
            # get best model
            load_best_model_at_end=True,
            metric_for_best_model="eval_macro_f1" if output_type == "cluster_prediction" else "eval_R²",
            greater_is_better = True
            
        )
           
        # define trainer
        if db.output_type == "cluster_prediction": 
            trainer = MyClassificationTrainer(
                model_init=model_init,
                args=args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                compute_metrics=compute_classification_metrics
                )  
        else:
            trainer = MyRegressionTrainer(
            model_init=model_init,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            compute_metrics=compute_regression_metrics)  
        
        wandb.init(project="bertology_acl", entity="zero0", name = "_".join([config_name, input_type ]))
        training_start = datetime.now()
        trainer.train()
        
        save_gpu_training_time(training_start, f"{lc_path}/probing_results/training_time.txt")
        runtime_minutes = round((datetime.now() - training_start).seconds /60, 2)
        
                             
        results = {"Dataset":dataset_name,"Model": model_name, "Layer": layer,"Filter": filter_name,
                      "Input": input_type, "Output": target_info.split("-")[0]}
  
        
        for split_name, split in {"train":train_dataset, "eval": eval_dataset, "test": test_dataset}.items():
                   
            split_outputs = trainer.predict(split, metric_key_prefix=split_name)
            split_results = split_outputs[2]
                   
            if output_type == "cluster_prediction":
                   
                results.update({f"{split_name} CrossEntropy": split_results[f"{split_name}_loss"], 
                        f"{split_name} Accuracy": split_results[f"{split_name}_accuracy"],
                        f"{split_name} Precision (Macro)": split_results[f"{split_name}_macro_precision"],
                        f"{split_name} Recall (Macro)": split_results[f"{split_name}_macro_recall"],
                        f"{split_name} F1 (Macro)": split_results[f"{split_name}_macro_f1"]
                       })
                   
                if split_name == "test":
                   # saving test predictions
                   predictions = split_outputs[0].argmax(axis=-1)
                   
                   with open(f'{save_path}probing_results/cluster_prediction/{config_name}_{input_type}_preds.npy', "wb") as f:
                       np.save(f, predictions)
                   
            else:
                   
                results.update({f"{split_name} MSE": split_results[f"{split_name}_loss"],
                                f"{split_name} R²": split_results[f"{split_name}_R²"],
                                f"{split_name} MAE": split_results[f"{split_name}_MAE"]
                              })
                   
                

        
        result_df = result_df.append(results, ignore_index = True)
        result_df.to_csv(save_path + "/probing_results/probing_results.csv")
    
        

        # saving model
        trainer.save_model(f'{save_path}model/{config_name}_{input_type}.pt')
                                                                                       
        # deleting checkpoints
        for direc in os.listdir(save_path + "checkpoint"):

            shutil.rmtree(save_path + "checkpoint/" + direc)                               
        
        wandb.finish()
    
        n_parameters = count_parameters(trainer.model)
        
        stats = stats.append({"Dataset":dataset_name,"Model": model_name, "Layer": layer,"Filter": filter_name,
                      "Input": input_type, "Output": target_info.split("-")[0],
                        "n_parameters": n_parameters,
                       "runtime (min)": runtime_minutes}, ignore_index = True)
                        

## Result Tables and Plots

In [ ]:
models = ["bert-base-uncased","microsoft-deberta-base"]
layers = ["layer6", "layer9", "layer12"]
column_order = []
datasets = ['semcor&omsti noun-synsets-strat', 'semcor&omsti person.n.01', 'cctweets-random None', 'cctweets-activist None']
output_types = ["kmeans2", "kmeans5", "pc1", "pc2"]


plt.rc('axes', labelsize=12)
plt.rc('axes', titlesize=12)  
plt.rc("font", size = 10)


tab20c = list(plt.cm.get_cmap("tab20c").colors)
c_map = {"BOW": tab20c[16], "POS": tab20c[17], "pos": tab20c[18]}

for dataset in datasets:
    
    for output_type in output_types:
        
        column_order.append((dataset, output_type))
        

for model in models:
    
    for layer in layers:
        
        selected_results = result_df.loc[(result_df["Model"] == model) &  (result_df["Layer"] == layer),
                                         ["Dataset", "Filter", "Input", "Output", 
                                         "test F1 (Macro)", "test R²"]]
        
        selected_results["Dataset"] = selected_results["Dataset"] + " " + selected_results["Filter"]
        
        selected_results["Objective"] = selected_results["test F1 (Macro)"].where(selected_results["test R²"].isna(),selected_results["test R²"])

        
        selected_results.drop(["Filter", "test F1 (Macro)","test R²"] , axis = 1, inplace = True)
        
        selected_results = selected_results.pivot(index = ["Input"], columns = ["Dataset", "Output"], values = "Objective")
        
        
        selected_results = selected_results[column_order]
        
        selected_results = selected_results.round(decimals = 2)
        
        selected_results = selected_results.rename(index={'bow_simple': 'BOW', "syntax": "POS", "position": "pos."})
        
        selected_results = selected_results.loc[["BOW", "POS", "pos."]]
        
        selected_results.to_csv(f"{save_path}probing_results/{model}_{layer}_probing_results.csv", sep = "&", line_terminator = "\\\\\n")
        
        plot_rows = 2
        plot_cols = 4
        
        
        png, png_axs = plt.subplots(plot_rows,plot_cols, figsize = (3.5*plot_cols,4*1))
    
        plot_col = 0
        
        
        # cols
        for dataset in ["semcor&omsti noun-synsets-strat", "semcor&omsti person.n.01", 
                        "cctweets-random None", "cctweets-activist None"]:
            
        
            dataset_df = selected_results[dataset]
            plot_row = 0
            
            #rows
            for output_type in ["pc", "kmeans"]:
                
                type_df = dataset_df[[col for col in dataset_df.columns if output_type in col]]
                
                
                BOW = type_df.loc["BOW"]
                POS = type_df.loc["POS"]
                pos = type_df.loc["pos."]
                
                x = np.arange(len(BOW))
                labels = BOW.index
                width = 0.2
                
                if plot_row == 0:
                    
                    title = dataset if dataset != "semcor&omsti noun-synsets-strat" else "semcor&omsti noun-synsets"
                    title = re.sub(" None", "", title)
                    png_axs[plot_row, plot_col].set_title(title, fontfamily= "serif")
            
                if plot_col == 0 and plot_row == 0:
                    
                    png_axs[plot_row, plot_col].set_ylabel(f"layer {layer[5:]}", fontfamily= "serif", ha = "right")
            
                png_axs[plot_row, plot_col].bar(x - width, BOW, width, label='BOW', color = c_map["BOW"])
                png_axs[plot_row, plot_col].bar(x , POS, width, label='POS', color = c_map["POS"])
                png_axs[plot_row, plot_col].bar(x + width, pos, width, label='pos.', color = c_map["pos"])
                
                png_axs[plot_row, plot_col].set_ylim(0, 1)
                png_axs[plot_row, plot_col].set_xticks(x)
                png_axs[plot_row, plot_col].set_xticklabels(labels)
                
                if plot_col == 3 and plot_row == 0:
                    
                    png_axs[plot_row, plot_col].legend(loc = "upper center")
                
                
                plt.subplots_adjust(hspace = .3)
                
                plot_row +=1
                
            plot_col +=1
            
        
        png.savefig(f'{save_path}plot/results_{model}_{layer}.png', bbox_inches = "tight")
            
        